In [2]:
import numpy as np
import matplotlib.pyplot as plt
from Tools.data_utils import SimulationData, lorenz_stochastic, generate_input_filename
from Tools.mathematical_tools import mutual_information, false_nearest_neighbors, denormalize_columns
from Tools.analysis_tools import reconstruct_signal_from_embedding

In [2]:
tau = 15
embedding_dim = 30
r = 20
alpha = 0
beta = 0
simulation_name= generate_input_filename(tau, embedding_dim, r, alpha, beta)    
# Carica i dati di simulazione
simulation = SimulationData.load_from_params(
    tau=tau, embedding_dim=embedding_dim, r=r, alpha=alpha, beta=beta, input_path="Simulation data"
)

z1 = simulation.trajectory[:, 0]
z2 = simulation.trajectory[:, 1]
z3 = simulation.trajectory[:, 2]

In [ ]:
# Grafico delle traiettorie 
fig = plt.figure(figsize=(12, 6))
# Primo subplot
ax = fig.add_subplot(121, projection="3d")
ax.plot(simulation.trajectory[:, 0], simulation.trajectory[:, 1], simulation.trajectory[:, 2], color='purple', lw=0.5)
ax.set_title("Attrattore di Lorenz Simulato")
ax.set_xlabel(f"$z_1$")
ax.set_ylabel(f"$z_2$")
ax.set_zlabel(f"$z_3$")
plt.show()


In [ ]:
# Grafico delle serie temporali
fig = plt.figure(figsize=(12, 6))
# Primo subplot
ax1 = fig.add_subplot(311)
ax1.plot(simulation.y1, color='blue', lw=0.5)
ax1.set_title(r"Serie Temporale $y_1 = z_2 + \beta \cdot \sigma$")
ax1.set_xlabel("Tempo")
ax1.set_ylabel("$z_1$")

ax2 = fig.add_subplot(312)
ax2.plot(simulation.y2[:,0], color='green', lw=0.5)
ax2.set_title(r"Serie Temporale $y_2(1) = z_1 + \beta \cdot \sigma$")
ax2.set_xlabel("Tempo")
ax2.set_ylabel("$z_2$")

ax3 = fig.add_subplot(313)
ax3.plot(simulation.y2[:,1], color='red', lw=0.5)
ax3.set_title(r"Serie Temporale $y_2(2) = z_3 + \beta \cdot \sigma$")
ax3.set_xlabel("Tempo")
ax3.set_ylabel("$z_3$")

plt.tight_layout()
plt.show()

In [5]:
# Inizializzazione della lista per salvare i risultati
range_f = 20  # Numero massimo di ritardi da analizzare
n_bins = 50   # Numero di bin per la discretizzazione
datDelayInformation_1 = []  
datDelayInformation_2 = []
datDelayInformation_3 = []

# Calcolo della mutual information per ogni ritardo
for delay in range(1, range_f):
    I = mutual_information(z1, delay, n_bins)  # mutual_information calcolata per z1
    datDelayInformation_1.append(I)

# Calcolo della mutual information per ogni ritardo
for delay in range(1, range_f):
    I = mutual_information(z2, delay, n_bins)  # mutual_information calcolata per z2
    datDelayInformation_2.append(I)

# Calcolo della mutual information per ogni ritardo
for delay in range(1, range_f):
    I = mutual_information(z3, delay, n_bins)  # mutual_information calcolata per z3
    datDelayInformation_3.append(I)

In [ ]:
# Grafico dei risultati
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

# Plot for datDelayInformation_1
axs[0].plot(range(1, range_f), datDelayInformation_1, color='purple')
axs[0].set_xlabel('Ritardo (delay)', fontsize=12)
axs[0].set_ylabel('Mutual Information', fontsize=12)
axs[0].set_title('Mutual Information vs Ritardo (z1)', fontsize=14)
axs[0].grid(True, linestyle='--', alpha=0.6)

# Plot for datDelayInformation_2
axs[1].plot(range(1, range_f), datDelayInformation_2, color='orange')
axs[1].set_xlabel('Ritardo (delay)', fontsize=12)
axs[1].set_ylabel('Mutual Information', fontsize=12)
axs[1].set_title('Mutual Information vs Ritardo (z2)', fontsize=14)
axs[1].grid(True, linestyle='--', alpha=0.6)

# Plot for datDelayInformation_3
axs[2].plot(range(1, range_f), datDelayInformation_3, color='green')
axs[2].set_xlabel('Ritardo (delay)', fontsize=12)
axs[2].set_ylabel('Mutual Information', fontsize=12)
axs[2].set_title('Mutual Information vs Ritardo (z3)', fontsize=14)
axs[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Parametri
embedding_dimensions = range(1, 10)  # Dimensioni di embedding da analizzare
delay = 15  # Ritardo temporale
threshold = 10  # Soglia per definire i falsi vicini

# Esegui il calcolo della frazione di falsi vicini per i tuoi dati
false_neighbors_fractions = [
    false_nearest_neighbors(z2, delay, embedding_dimension=d, threshold=threshold)
    for d in embedding_dimensions
]

# Filtra eventuali valori None (se i dati sono insufficienti)
false_neighbors_fractions = [fnn for fnn in false_neighbors_fractions if fnn is not None]

# Grafico
plt.figure(figsize=(10, 6))
plt.plot(embedding_dimensions[:len(false_neighbors_fractions)], false_neighbors_fractions, marker='o', color='purple')
plt.xlabel('Dimensione di Embedding', fontsize=12)
plt.ylabel('Frazione di Falsi Vicini', fontsize=12)
plt.title('Falsi Vicini vs Dimensione di Embedding', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
from numpy.linalg import pinv

# Original Hankel matrix
original_input_H1 = simulation.H1 #This is not modified by normalization

# Denormalization check
after = denormalize_columns(simulation.projected_H1, simulation.min_h1, simulation.max_h1)

# Calcola la pseudo-inversa di P
P_pseudo_inverse = pinv(simulation.P)

print("Pseudo-inverse shape (rxd): ", P_pseudo_inverse.shape)
print("Projected Hankel shape(rx n_sample): ", after.shape)

decompr_H1 = np.dot(after.T,P_pseudo_inverse)
decompr_H1 = decompr_H1.T
print("Decompressed Hankel shape: ", decompr_H1.shape)

# Test difference between original and decompressed Hankel
diffDeco = np.sum(np.abs(original_input_H1 - decompr_H1))
print("Difference between original and decompressed Hankel (Inverse MAtrix): ", diffDeco)

decompressed_signal = reconstruct_signal_from_embedding(decompr_H1.T,simulation.embedding_dim, simulation.tau)
original_signal = reconstruct_signal_from_embedding(original_input_H1.T,simulation.embedding_dim, simulation.tau)

# Calculate MSE and R²
mse1 = np.mean((original_signal - decompressed_signal) ** 2)
r2_1 = 1 - np.sum((original_signal - decompressed_signal) ** 2) / np.sum((original_signal - np.mean(original_signal)) ** 2)

# Plot dei segnali
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(original_signal, label="z2", color='blue', lw=0.5)
ax.plot(decompressed_signal, label="z2 recon", color='orange', lw=0.5)
ax.legend()
ax.set_title(r"$Z_2$ recon: MSE = {:.4f}, R² = {:.4f}".format(mse1, r2_1))
ax.set_xlabel("Tempo")
ax.set_ylabel("$z_2$")

plt.tight_layout()
plt.show()
